<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="300" alt="Skills Network Logo">
    </a>
</p>


# Test Environment for Generative AI classroom labs

This lab provides a test environment for the codes generated using the Generative AI classroom.

Follow the instructions below to set up this environment for further use.


# Setup


### Install required libraries

In case of a requirement of installing certain python libraries for use in your task, you may do so as shown below.


In [1]:
%pip install seaborn
import piplite

await piplite.install(['nbformat', 'plotly'])

### Dataset URL from the GenAI lab
Use the URL provided in the GenAI lab in the cell below. 


In [2]:
URL =  "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DA0101EN-Coursera/laptop_pricing_dataset_mod1.csv"

### Downloading the dataset

Execute the following code to download the dataset in to the interface.

> Please note that this step is essential in JupyterLite. If you are using a downloaded version of this notebook and running it on JupyterLabs, then you can skip this step and directly use the URL in pandas.read_csv() function to read the dataset as a dataframe


In [3]:
from pyodide.http import pyfetch

async def download(url, filename):
    response = await pyfetch(url)
    if response.status == 200:
        with open(filename, "wb") as f:
            f.write(await response.bytes())

path = URL

await download(path, "dataset.csv")
file_name  = "dataset.csv"

---


# Test Environment


In [4]:
# Keep appending the code generated to this cell, or add more cells below this to execute in parts
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler

<ipython-input-4-ccf89c1cb302>:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [5]:
def preprocess_dataframe(df, target_col='Price', drop_index_col=True, verbose=False):
    """
    Returns:
      - df_std: features transformed with StandardScaler for numeric cols (and OneHotEncoded categoricals)
      - df_minmax: features transformed with MinMaxScaler for numeric cols (and OneHotEncoded categoricals)
      - y: target values if target_col exists in df, else None
    """
    df = df.copy()

    # Drop the index-like column if present
    if drop_index_col and 'Unnamed: 0' in df.columns:
        df = df.drop(columns=['Unnamed: 0'])
        if verbose:
            print("Dropped column 'Unnamed: 0'")

    # Separate target if present
    if target_col in df.columns:
        y = df[target_col]
        X = df.drop(columns=[target_col])
        if verbose:
            print(f"Target column '{target_col}' found and separated from features.")
    else:
        y = None
        X = df
        if verbose:
            print(f"Target column '{target_col}' not found. Using all columns as features.")

    # Identify categorical vs numeric columns
    categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
    numeric_cols = [c for c in X.columns if c not in categorical_cols]

    if verbose:
        print(f"Categorical columns: {categorical_cols}")
        print(f"Numeric columns: {numeric_cols}")

    # Preprocessing for numeric features: impute then scale
    numeric_imputer = SimpleImputer(strategy='median')
    std_scaler = StandardScaler()
    minmax_scaler = MinMaxScaler()
    std_num = Pipeline([('imputer', numeric_imputer),
                        ('scaler', std_scaler)])
    minmax_num = Pipeline([('imputer', numeric_imputer),
                           ('scaler', minmax_scaler)])

    # Preprocessing for categorical features: impute then one-hot encode
    categorical_imputer = SimpleImputer(strategy='most_frequent')
    onehot = OneHotEncoder(handle_unknown='ignore')
    cat_pipe = Pipeline([('imputer', categorical_imputer),
                        ('onehot', onehot)])

    # Two separate transformers: one with StandardScaler, one with MinMaxScaler
    preprocessor_std = ColumnTransformer(
        transformers=[
            ('num', std_num, numeric_cols),
            ('cat', cat_pipe, categorical_cols)
        ]
    )

    preprocessor_minmax = ColumnTransformer(
        transformers=[
            ('num', minmax_num, numeric_cols),
            ('cat', cat_pipe, categorical_cols)
        ]
    )

    # Fit/transform on the features
    X_std = preprocessor_std.fit_transform(X)
    X_minmax = preprocessor_minmax.fit_transform(X)

    # Get feature names for the transformed data (for nicer DataFrames)
    # Note: get_feature_names_out is supported by scikit-learn >= 0.23
    names_std = preprocessor_std.get_feature_names_out(input_features=X.columns)
    names_minmax = preprocessor_minmax.get_feature_names_out(input_features=X.columns)

    df_std = pd.DataFrame(X_std, columns=names_std, index=X.index)
    df_minmax = pd.DataFrame(X_minmax, columns=names_minmax, index=X.index)
    return df_std, df_minmax, y

In [7]:
df = pd.read_csv('dataset.csv')
df_std, df_minmax, y = preprocess_dataframe(df, target_col='Price', drop_index_col=True, verbose=True)

print("Standardized features shape:", df_std.shape)
print("Min-Max scaled features shape:", df_minmax.shape)
df_std.to_csv('data_std.csv', index=False)
df_minmax.to_csv('data_minmax.csv', index=False)

Dropped column 'Unnamed: 0'
Target column 'Price' found and separated from features.
Categorical columns: ['Manufacturer', 'Screen']
Numeric columns: ['Category', 'GPU', 'OS', 'CPU_core', 'Screen_Size_cm', 'CPU_frequency', 'RAM_GB', 'Storage_GB_SSD', 'Weight_kg']
Standardized features shape: (238, 22)
Min-Max scaled features shape: (238, 22)


In [8]:
import pandas as pd

def read_csv_with_headers(path, header_rows=1, sep=',', encoding=None, flatten_multiheader=False):
    """
    Read a CSV into a DataFrame with configurable header rows.
    - header_rows = 1 -> first row is the header (default)
    - header_rows > 1 -> first 'header_rows' rows form a MultiIndex header
    - flatten_multiheader: if True and a MultiIndex header exists, flatten to single level
    """
    if header_rows < 1:
        raise ValueError("header_rows must be >= 1")

    header = 0 if header_rows == 1 else list(range(header_rows))
    df = pd.read_csv(path, header=header, sep=sep, encoding=encoding)

    if flatten_multiheader and isinstance(df.columns, pd.MultiIndex):
        df.columns = ['_'.join([str(c) for c in tup if c is not None]) for tup in df.columns.values]

    return df

In [9]:
def missing_values_per_column(df: pd.DataFrame) -> pd.DataFrame:
    """
    Return a DataFrame listing columns that have missing values, with:
    - column: column name
    - n_missing: number of missing values in the column
    - pct_missing: percentage of missing values
    - dtype: data type of the column

    Only columns with at least one missing value are included.
    """
    # counts of missing values per column
    n_missing = df.isnull().sum()
    total = len(df)

    # build result DataFrame
    result = pd.DataFrame({
        'column': df.columns,
        'n_missing': n_missing.values,
        'pct_missing': (n_missing.values / total * 100).round(2),
        'dtype': df.dtypes.values
    })

    # filter to columns with any missing values and sort by missing count
    result = result[result['n_missing'] > 0].sort_values(by='n_missing', ascending=False).reset_index(drop=True)
    return result

In [11]:
df = read_csv_with_headers('dataset.csv')

In [12]:
print(missing_values_per_column(df))

           column  n_missing  pct_missing    dtype
0       Weight_kg          5         2.10  float64
1  Screen_Size_cm          4         1.68  float64


## Authors


[Abhishek Gagneja](https://www.linkedin.com/in/abhishek-gagneja-23051987/)


## Change Log


|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2023-12-10|0.1|Abhishek Gagneja|Initial Draft created|


Copyright © 2023 IBM Corporation. All rights reserved.
